# 📊 Logistic Regression & TF-IDF Ensemble - Smart MCQ Solver (With Advanced Wandb Logging)

### 📌 Overview & Wandb Tracking Features
This notebook implements the complete **Logistic Regression + Ridge + SGD TF-IDF Ensemble** with **Rich Fold-Wise and Metric-Wise Weights & Biases (Wandb) Logging**.

- **Primary Kaggle Metric**: `MAP@3` (Mean Average Precision at 3)
- **Secondary Evaluation Metrics**:
  - `Top-1 Accuracy`: Standard classification accuracy (single best prediction).
  - `Top-3 Accuracy`: Recall metric checking if the ground truth is within top 3 ranked answers.
  - `F1-Score (Macro)`: Harmonic mean of precision and recall.
  - `Precision (Macro)`: Proportion of correct positive predictions.
  - `Recall (Macro)`: Coverage of actual ground truth labels.
  - `Log Loss`: Binary Cross Entropy of option probability predictions.

- **Wandb Dashboard Logs**:
  - **Hyperparameter Configuration**: Track C, alpha, TF-IDF max_features, n-gram ranges, n_splits.
  - **Per-Fold Metric Progression**: Logs `fold/val_loss`, `fold/val_map3`, `fold/val_top1_accuracy`, `fold/val_top3_accuracy`, `fold/val_f1_score`, `fold/val_precision`, `fold/val_recall`.
  - **Comparative Model Summary Table**: Automatically generates a `wandb.Table` comparing Logistic Regression, Ridge Classifier, SGD Classifier, and Ensemble Blend.

---


In [1]:
# Cell 1: Core Imports & Dependencies
import os
import re
import warnings
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, log_loss

warnings.filterwarnings('ignore')
print("Core libraries imported successfully!")


Core libraries imported successfully!


In [2]:
# Cell 2: Weights & Biases (Wandb) Authentication & Setup
try:
    import wandb
    WANDB_API_KEY = "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"
    WANDB_ENTITY = "models-indian-institute-of-technology-madras481" 
    WANDB_PROJECT = "23f3001092-t22026"

    wandb.login(key=WANDB_API_KEY)
    USE_WANDB = True
    print("Wandb initialized successfully!")
except Exception as e:
    USE_WANDB = False
    print(f"Wandb running in offline fallback mode ({e})")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Wandb initialized successfully!


In [3]:
# Cell 3: Dataset Loading & Setup
def load_datasets():
    train_candidates = [
        "train.csv",
        "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",
        "/kaggle/input/smart-mcq-solver-challenge/train.csv"
    ]
    test_candidates = [
        "test.csv",
        "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv",
        "/kaggle/input/smart-mcq-solver-challenge/test.csv"
    ]
    
    train_path = next((p for p in train_candidates if os.path.exists(p)), train_candidates[0])
    test_path = next((p for p in test_candidates if os.path.exists(p)), test_candidates[0])
    
    df_train = pd.read_csv(train_path).fillna('')
    df_test = pd.read_csv(test_path).fillna('')
    
    return df_train, df_test

train, test = load_datasets()
OPTION_COLS = ['A', 'B', 'C', 'D', 'E']

print(f"Train shape: {train.shape}")
print(f"Test shape : {test.shape}")
train.head(3)


Train shape: (2000, 8)
Test shape : (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [4]:
# Cell 4: Comprehensive Metric Evaluation Functions
def apk(actual, predicted, k=3):
    if actual in predicted[:k]:
        return 1.0 / (predicted[:k].index(actual) + 1.0)
    return 0.0

def mapk(actuals, preds, k=3):
    return float(np.mean([apk(a, p, k) for a, p in zip(actuals, preds)]))

def compute_all_metrics(actuals, pred_lists):
    """
    Computes MAP@3, Top-1 Accuracy, Top-3 Recall Accuracy, F1-Score, Precision, and Recall.
    """
    map3_score = mapk(actuals, pred_lists)
    
    top1_preds = [p[0] for p in pred_lists]
    top1_acc = accuracy_score(actuals, top1_preds)
    
    top3_acc = np.mean([1 if a in p[:3] else 0 for a, p in zip(actuals, pred_lists)])
    
    prec, rec, f1, _ = precision_recall_fscore_support(actuals, top1_preds, average='macro', zero_division=0)
    
    return {
        'map3': map3_score,
        'top1_accuracy': top1_acc,
        'top3_accuracy': top3_acc,
        'f1_score': f1,
        'precision': prec,
        'recall': rec
    }

def rank_preds(pw, scores, orig_df):
    pw_temp = pw.copy()
    pw_temp['score'] = scores
    
    ranked = pw_temp.sort_values(['qid', 'score'], ascending=[True, False])
    grouped = ranked.groupby('qid')['option'].apply(lambda opts: list(opts)[:3]).to_dict()
    
    return [grouped.get(qid, ['A', 'B', 'C']) for qid in orig_df['id']]


In [5]:
# Cell 5: Pairwise Feature Engineering (Word Overlap + Jaccard + Length Ratio)
def make_pairwise_dataset(df, is_train=True):
    rows = []
    for _, r in df.iterrows():
        qid = r['id']
        prompt = str(r['prompt'])
        p_words = set(re.findall(r'\w+', prompt.lower()))
        ans = r.get('answer', None)
        
        for c in OPTION_COLS:
            opt_str = str(r[c])
            o_words = set(re.findall(r'\w+', opt_str.lower()))
            
            overlap = len(p_words.intersection(o_words))
            jaccard = overlap / max(len(p_words.union(o_words)), 1)
            len_ratio = len(opt_str) / max(len(prompt), 1)
            
            combined_text = f"{prompt} [SEP] {opt_str}"
            
            row = {
                'qid': qid,
                'option': c,
                'text': combined_text,
                'overlap': overlap,
                'jaccard': jaccard,
                'len_ratio': len_ratio
            }
            if is_train:
                row['label'] = 1 if (ans == c) else 0
            rows.append(row)
            
    return pd.DataFrame(rows)

pw_train = make_pairwise_dataset(train, is_train=True)
pw_test = make_pairwise_dataset(test, is_train=False)

print(f"Pairwise Train Shape: {pw_train.shape}")
print(f"Pairwise Test Shape : {pw_test.shape}")
pw_train.head(3)


Pairwise Train Shape: (10000, 7)
Pairwise Test Shape : (2500, 6)


,qid,option,text,overlap,jaccard,len_ratio,label
0,1,A,Pick the best possible answer: What is Martin ...,8,0.153846,2.161972,0
1,1,B,Pick the best possible answer: What is Martin ...,7,0.148936,1.816901,1
2,1,C,Pick the best possible answer: What is Martin ...,10,0.250000,1.563380,0


In [6]:
# Cell 6: TF-IDF Sparse Feature Matrix Construction
print("Fitting Word & Character TF-IDF Vectorizers...")

tfidf_word = TfidfVectorizer(max_features=80000, ngram_range=(1, 3), sublinear_tf=True, analyzer='word')
tfidf_char = TfidfVectorizer(max_features=50000, ngram_range=(2, 5), sublinear_tf=True, analyzer='char_wb')

combined_corpus = pd.concat([pw_train['text'], pw_test['text']])
tfidf_word.fit(combined_corpus)
tfidf_char.fit(combined_corpus)

Xw_tr = tfidf_word.transform(pw_train['text'])
Xw_te = tfidf_word.transform(pw_test['text'])

Xc_tr = tfidf_char.transform(pw_train['text'])
Xc_te = tfidf_char.transform(pw_test['text'])

X_num_tr = csr_matrix(pw_train[['overlap', 'jaccard', 'len_ratio']].values)
X_num_te = csr_matrix(pw_test[['overlap', 'jaccard', 'len_ratio']].values)

X_tr = hstack([Xw_tr, Xc_tr, X_num_tr])
X_te = hstack([Xw_te, Xc_te, X_num_te])

y_tr = pw_train['label'].values
groups = pw_train['qid'].values

print(f"Train Feature Matrix: {X_tr.shape}")
print(f"Test Feature Matrix : {X_te.shape}")


Fitting Word & Character TF-IDF Vectorizers...
Train Feature Matrix: (10000, 53432)
Test Feature Matrix : (2500, 53432)


In [7]:
# Cell 7: 5-Fold Training Loop with Detailed Fold-Wise & Model-Wise Wandb Logging
gkf = GroupKFold(n_splits=5)

oof_lr = np.zeros(len(pw_train))
oof_ridge = np.zeros(len(pw_train))
oof_sgd = np.zeros(len(pw_train))

tst_lr = np.zeros(len(pw_test))
tst_ridge = np.zeros(len(pw_test))
tst_sgd = np.zeros(len(pw_test))

config_dict = {
    "architecture": "Logistic Regression + Ridge + SGD Ensemble",
    "dataset": "Smart MCQ Solver Challenge",
    "C_lr": 6.0,
    "alpha_ridge": 1.0,
    "loss_sgd": "log_loss",
    "max_features_word": 80000,
    "max_features_char": 50000,
    "ngram_range_word": "(1,3)",
    "ngram_range_char": "(2,5)",
    "n_splits": 5,
    "ensemble_weights": {"LR": 0.50, "Ridge": 0.30, "SGD": 0.20}
}

if USE_WANDB:
    run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name="scrach-lr-detailed-fold-logs",
        config=config_dict
    )

print("\nStarting 5-Fold Cross Validation...")

fold_metrics_list = []

for fold, (ti, vi) in enumerate(gkf.split(X_tr, y_tr, groups)):
    print(f"\n--- Fold {fold+1}/5 ---")
    
    # 1. Logistic Regression
    lr = LogisticRegression(C=6.0, max_iter=2000)
    lr.fit(X_tr[ti], y_tr[ti])
    oof_lr[vi] = lr.predict_proba(X_tr[vi])[:, 1]
    tst_lr += lr.predict_proba(X_te)[:, 1] / 5.0
    
    # 2. Ridge Classifier
    ridge = RidgeClassifier(alpha=1.0)
    ridge.fit(X_tr[ti], y_tr[ti])
    oof_ridge[vi] = ridge.decision_function(X_tr[vi])
    tst_ridge += ridge.decision_function(X_te) / 5.0
    
    # 3. SGD Classifier
    sgd = SGDClassifier(loss='log_loss', max_iter=1000, random_state=42)
    sgd.fit(X_tr[ti], y_tr[ti])
    oof_sgd[vi] = sgd.predict_proba(X_tr[vi])[:, 1]
    tst_sgd += sgd.predict_proba(X_te)[:, 1] / 5.0
    
    # Fold Blend Predictions
    fold_blend_scores = 0.50 * oof_lr[vi] + 0.30 * oof_ridge[vi] + 0.20 * oof_sgd[vi]
    
    val_qids = pw_train.iloc[vi]['qid'].unique()
    val_df_orig = train[train['id'].isin(val_qids)].reset_index(drop=True)
    val_pw_df = pw_train[pw_train['qid'].isin(val_qids)].reset_index(drop=True)
    
    fold_preds = rank_preds(val_pw_df, fold_blend_scores, val_df_orig)
    fold_actuals = val_df_orig['answer'].tolist()
    
    # Compute Fold Metrics
    m = compute_all_metrics(fold_actuals, fold_preds)
    fold_val_loss = log_loss(val_pw_df['label'], fold_blend_scores)
    
    print(f" Fold {fold+1} Loss: {fold_val_loss:.4f} | MAP@3: {m['map3']:.4f} | Acc: {m['top1_accuracy']:.4f} | F1: {m['f1_score']:.4f}")
    
    fold_metrics_list.append(m)
    
    # Log Fold Metrics to Wandb
    if USE_WANDB:
        wandb.log({
            "fold": fold + 1,
            "fold/val_loss": fold_val_loss,
            "fold/val_map3": m['map3'],
            "fold/val_top1_accuracy": m['top1_accuracy'],
            "fold/val_top3_accuracy": m['top3_accuracy'],
            "fold/val_f1_score": m['f1_score'],
            "fold/val_precision": m['precision'],
            "fold/val_recall": m['recall']
        })

# Overall Out-Of-Fold Predictions
oof_blend = 0.50 * oof_lr + 0.30 * oof_ridge + 0.20 * oof_sgd
tst_blend = 0.50 * tst_lr + 0.30 * tst_ridge + 0.20 * tst_sgd

overall_actuals = train['answer'].tolist()

# Overall Model Metrics
m_lr = compute_all_metrics(overall_actuals, rank_preds(pw_train, oof_lr, train))
m_ridge = compute_all_metrics(overall_actuals, rank_preds(pw_train, oof_ridge, train))
m_sgd = compute_all_metrics(overall_actuals, rank_preds(pw_train, oof_sgd, train))
m_blend = compute_all_metrics(overall_actuals, rank_preds(pw_train, oof_blend, train))

oof_loss = log_loss(pw_train['label'], oof_blend)

print("\n==================================================")
print("       OVERALL OUT-OF-FOLD PERFORMANCE SUMMARY")
print("==================================================")
print(f" Primary Kaggle Metric (MAP@3): {m_blend['map3']:.4f}")
print(f" Top-1 Accuracy               : {m_blend['top1_accuracy']:.4f}")
print(f" Top-3 Recall Accuracy        : {m_blend['top3_accuracy']:.4f}")
print(f" Macro F1 Score               : {m_blend['f1_score']:.4f}")
print(f" Macro Precision              : {m_blend['precision']:.4f}")
print(f" Macro Recall                 : {m_blend['recall']:.4f}")
print(f" Pairwise Log Loss            : {oof_loss:.4f}")
print("==================================================\n")

# Log Overall Summary & Comparative Table to Wandb
if USE_WANDB:
    wandb.log({
        "oof/overall_map3": m_blend['map3'],
        "oof/overall_top1_accuracy": m_blend['top1_accuracy'],
        "oof/overall_top3_accuracy": m_blend['top3_accuracy'],
        "oof/overall_f1_score": m_blend['f1_score'],
        "oof/overall_precision": m_blend['precision'],
        "oof/overall_recall": m_blend['recall'],
        "oof/overall_val_loss": oof_loss,
        "models/lr_map3": m_lr['map3'],
        "models/ridge_map3": m_ridge['map3'],
        "models/sgd_map3": m_sgd['map3'],
    })
    
    # Create Wandb Summary Table
    table = wandb.Table(columns=["Model Architecture", "MAP@3 Score", "Top-1 Accuracy", "Top-3 Accuracy", "F1 Score", "Precision", "Recall"])
    table.add_data("Logistic Regression (C=6.0)", m_lr['map3'], m_lr['top1_accuracy'], m_lr['top3_accuracy'], m_lr['f1_score'], m_lr['precision'], m_lr['recall'])
    table.add_data("Ridge Classifier (alpha=1.0)", m_ridge['map3'], m_ridge['top1_accuracy'], m_ridge['top3_accuracy'], m_ridge['f1_score'], m_ridge['precision'], m_ridge['recall'])
    table.add_data("SGD Classifier (log_loss)", m_sgd['map3'], m_sgd['top1_accuracy'], m_sgd['top3_accuracy'], m_sgd['f1_score'], m_sgd['precision'], m_sgd['recall'])
    table.add_data("Ensemble Blend (LR+Ridge+SGD)", m_blend['map3'], m_blend['top1_accuracy'], m_blend['top3_accuracy'], m_blend['f1_score'], m_blend['precision'], m_blend['recall'])
    
    wandb.log({"model_comparative_summary": table})
    wandb.finish()


wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260727_172729-l7l4xdyg
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run scrach-lr-detailed-fold-logs
wandb: ⭐️ View project at https://wandb.ai/models-indian-institute-of-technology-madras481/23f3001092-t22026
wandb: 🚀 View run at https://wandb.ai/models-indian-institute-of-technology-madras481/23f3001092-t22026/runs/l7l4xdyg



Starting 5-Fold Cross Validation...

--- Fold 1/5 ---
 Fold 1 Loss: 0.1539 | MAP@3: 0.9971 | Acc: 0.9950 | F1: 0.9946

--- Fold 2/5 ---
 Fold 2 Loss: 0.1190 | MAP@3: 0.9962 | Acc: 0.9925 | F1: 0.9924

--- Fold 3/5 ---
 Fold 3 Loss: 0.1142 | MAP@3: 0.9988 | Acc: 0.9975 | F1: 0.9974

--- Fold 4/5 ---
 Fold 4 Loss: 0.1191 | MAP@3: 0.9975 | Acc: 0.9950 | F1: 0.9952

--- Fold 5/5 ---
 Fold 5 Loss: 0.1433 | MAP@3: 0.9975 | Acc: 0.9950 | F1: 0.9952

       OVERALL OUT-OF-FOLD PERFORMANCE SUMMARY
 Primary Kaggle Metric (MAP@3): 0.9974
 Top-1 Accuracy               : 0.9950
 Top-3 Recall Accuracy        : 1.0000
 Macro F1 Score               : 0.9950
 Macro Precision              : 0.9957
 Macro Recall                 : 0.9944
 Pairwise Log Loss            : 0.1299



wandb: uploading artifact run-l7l4xdyg-model_comparative_summary; updating run metadata
wandb: uploading artifact run-l7l4xdyg-model_comparative_summary
wandb: uploading history steps 4-6, summary, console lines 16-28
wandb: 
wandb: Run history:
wandb:                   fold ▁▃▅▆█
wandb:      fold/val_f1_score ▄▁█▅▅
wandb:          fold/val_loss █▂▁▂▆
wandb:          fold/val_map3 ▃▁█▅▅
wandb:     fold/val_precision ▄▁█▅▄
wandb:        fold/val_recall ▄▁█▅▆
wandb: fold/val_top1_accuracy ▅▁█▅▅
wandb: fold/val_top3_accuracy ▁▁▁▁▁
wandb:         models/lr_map3 ▁
wandb:      models/ridge_map3 ▁
wandb:                     +8 ...
wandb: 
wandb: Run summary:
wandb:                   fold 5
wandb:      fold/val_f1_score 0.9952
wandb:          fold/val_loss 0.1433
wandb:          fold/val_map3 0.9975
wandb:     fold/val_precision 0.99545
wandb:        fold/val_recall 0.99506
wandb: fold/val_top1_accuracy 0.995
wandb: fold/val_top3_accuracy 1
wandb:         models/lr_map3 0.99742
wandb:      mod

In [8]:
# Cell 8: Final Submission Generation & Comparative Report Table
test_preds = rank_preds(pw_test, tst_blend, test)

submission = pd.DataFrame({
    'ID': test['id'],
    'Prediction': [' '.join(p) for p in test_preds]
})

submission.to_csv('submission.csv', index=False)
print("Successfully generated submission.csv!")

assert len(submission) == len(test), f"Expected {len(test)} rows, got {len(submission)}"
assert submission['Prediction'].str.match(r'^[A-E] [A-E] [A-E]$').all(), "Invalid prediction format!"

# Print Comparative Summary Markdown Table for Project Report
report_df = pd.DataFrame([
    {"Model Architecture": "Logistic Regression (C=6.0)", "MAP@3": f"{m_lr['map3']:.4f}", "Top-1 Acc": f"{m_lr['top1_accuracy']:.4f}", "Top-3 Acc": f"{m_lr['top3_accuracy']:.4f}", "F1 Score": f"{m_lr['f1_score']:.4f}"},
    {"Model Architecture": "Ridge Classifier (alpha=1.0)", "MAP@3": f"{m_ridge['map3']:.4f}", "Top-1 Acc": f"{m_ridge['top1_accuracy']:.4f}", "Top-3 Acc": f"{m_ridge['top3_accuracy']:.4f}", "F1 Score": f"{m_ridge['f1_score']:.4f}"},
    {"Model Architecture": "SGD Classifier (log_loss)", "MAP@3": f"{m_sgd['map3']:.4f}", "Top-1 Acc": f"{m_sgd['top1_accuracy']:.4f}", "Top-3 Acc": f"{m_sgd['top3_accuracy']:.4f}", "F1 Score": f"{m_sgd['f1_score']:.4f}"},
    {"Model Architecture": "🏆 Ensemble Blend (LR+Ridge+SGD)", "MAP@3": f"{m_blend['map3']:.4f}", "Top-1 Acc": f"{m_blend['top1_accuracy']:.4f}", "Top-3 Acc": f"{m_blend['top3_accuracy']:.4f}", "F1 Score": f"{m_blend['f1_score']:.4f}"}
])

print("\n" + "="*70)
print("          MODEL COMPARATIVE REPORT FOR REPORT DOCUMENTATION")
print("="*70)
print(report_df.to_string(index=False))
print("="*70 + "\n")

print("Submission Sample:")
submission.head(10)


Successfully generated submission.csv!

          MODEL COMPARATIVE REPORT FOR REPORT DOCUMENTATION
             Model Architecture  MAP@3 Top-1 Acc Top-3 Acc F1 Score
    Logistic Regression (C=6.0) 0.9974    0.9950    1.0000   0.9950
   Ridge Classifier (alpha=1.0) 0.9974    0.9950    1.0000   0.9950
      SGD Classifier (log_loss) 0.9928    0.9860    0.9995   0.9858
🏆 Ensemble Blend (LR+Ridge+SGD) 0.9974    0.9950    1.0000   0.9950

Submission Sample:


,ID,Prediction
0,1,A E D
1,2,B E C
2,3,B E D
3,4,E C D
4,5,C A D
5,6,D A C
6,7,E D C
7,8,B E A
8,9,C D E
9,10,B D C
